# Speech Emotion Fine-Tuning — Colab / Kaggle

Run speech emotion recognition experiments on RESD dataset.
Tested on T4 GPU (Colab free tier / Kaggle).

**Steps:**
1. Set up environment
2. Clone repo (or upload files)
3. Pick a config and train
4. Visualise results

## 1. GPU check

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU found — switch runtime to GPU.')

## 2. Install dependencies

In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'torchaudio', 'transformers>=4.40',
                'datasets>=2.18', 'peft>=0.10',
                'scikit-learn', 'matplotlib', 'seaborn',
                'soundfile', 'pyyaml', 'tqdm'], check=True)
print('Done.')

## 3. Get the code

**Option A — clone from GitHub** (replace with your repo URL):

In [ ]:
import os
REPO_URL = 'https://github.com/YOUR_USERNAME/speech_emo_finetune.git'  # <- edit this
REPO_DIR = 'speech_emo_finetune'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

**Option B — Google Drive mount** (Colab only):

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/speech_emo_finetune')

## 4. Choose experiment config

In [ ]:
# Available configs — pick one:
#   wavlm_base_head    (fastest, ~8 min on T4)
#   wavlm_base_lora    (~12 min)
#   wavlm_base_full    (~20 min)
#   hubert_large_head  (~15 min)
#   hubert_large_lora  (~25 min)
#   hubert_large_topn  (~25 min)
#   hubert_large_lstm  (~10 min, frozen backbone)
#   wav2vec2_xlsr_head (~15 min)
#   wav2vec2_xlsr_lora (~25 min)

CONFIG = 'configs/wavlm_base_head.yaml'
print('Selected config:', CONFIG)

## 5. Train

In [ ]:
subprocess.run([sys.executable, 'train.py', CONFIG], check=True)

## 6. Visualise training curves

In [ ]:
import json
import pathlib
import matplotlib.pyplot as plt

# Find output dir from config
import yaml
with open(CONFIG) as f:
    cfg = yaml.safe_load(f)
output_dir = pathlib.Path(cfg.get('output_dir', 'outputs/experiment'))
metrics_path = output_dir / 'metrics.jsonl'

records = []
with open(metrics_path) as f:
    for line in f:
        r = json.loads(line)
        if 'epoch' in r:  # skip test record
            records.append(r)

epochs = [r['epoch'] for r in records]
train_loss = [r['train_loss'] for r in records]
dev_f1 = [r['dev_f1_weighted'] for r in records]
dev_acc = [r['dev_accuracy'] for r in records]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_loss, marker='o')
axes[0].set_title('Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(epochs, dev_f1, marker='o', label='F1 weighted')
axes[1].plot(epochs, dev_acc, marker='s', linestyle='--', label='Accuracy')
axes[1].set_title('Dev Metrics')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.suptitle(cfg.get('run_name', CONFIG))
plt.tight_layout()
plt.show()

## 7. Test results + confusion matrix

In [ ]:
import numpy as np
import seaborn as sns

# Load last record (test split)
with open(metrics_path) as f:
    all_records = [json.loads(l) for l in f]
test_rec = [r for r in all_records if r.get('split') == 'test'][-1]

print(f"Test accuracy:          {test_rec['accuracy']:.4f}")
print(f"Test weighted_accuracy: {test_rec['weighted_accuracy']:.4f}")
print(f"Test F1 macro:          {test_rec['f1_macro']:.4f}")
print(f"Test F1 weighted:       {test_rec['f1_weighted']:.4f}")
print()
print(f"{'Class':12s} {'P':>6} {'R':>6} {'F1':>6} {'n':>5}")
for cls, vals in test_rec['per_class'].items():
    print(f"{cls:12s} {vals['precision']:6.3f} {vals['recall']:6.3f} {vals['f1']:6.3f} {int(vals['support']):5d}")

# Confusion matrix
cm = np.array(test_rec['confusion_matrix'])
label_names = list(test_rec['per_class'].keys())

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=label_names, yticklabels=label_names, ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — {cfg.get("run_name", "")}')
plt.tight_layout()
plt.show()

## 8. (Optional) Compare multiple experiments

In [ ]:
import pathlib

results = []
for metrics_file in sorted(pathlib.Path('outputs').rglob('metrics.jsonl')):
    with open(metrics_file) as f:
        recs = [json.loads(l) for l in f]
    test_recs = [r for r in recs if r.get('split') == 'test']
    if not test_recs:
        continue
    tr = test_recs[-1]
    run = metrics_file.parent.name
    results.append({
        'run': run,
        'accuracy': tr['accuracy'],
        'weighted_accuracy': tr['weighted_accuracy'],
        'f1_macro': tr['f1_macro'],
        'f1_weighted': tr['f1_weighted'],
    })

if results:
    import pandas as pd
    df = pd.DataFrame(results).set_index('run').sort_values('f1_weighted', ascending=False)
    display(df.style.format('{:.4f}').background_gradient(cmap='YlGn'))

    fig, ax = plt.subplots(figsize=(10, 5))
    df[['accuracy', 'f1_macro', 'f1_weighted']].plot(kind='bar', ax=ax)
    ax.set_title('Experiment comparison')
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.show()
else:
    print('No completed experiments found in outputs/')